In [1]:
from embedder import Embedder
embed = Embedder()

In [2]:
v = embed.encode(
    "How does approximate nearest neighbor search work?"
)
print(v[0])

-0.020582034180917443


In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [5]:
doc = None
for d in documents:
    if d["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md":
        doc = d
        break

In [6]:
doc["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [8]:
doc_vector = embed.encode(
    doc["content"]
)

similarity = v.dot(doc_vector)
print(similarity)

0.36107008472347096


In [9]:
from gitsource import chunk_documents

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

In [10]:
texts = [
    chunk["content"]
    for chunk in chunks
]

In [11]:
import numpy as np
X = embed.encode_batch(texts)
scores = X.dot(v)
idx = np.argmax(scores)
chunks[idx]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [12]:
from minsearch import VectorSearch

vindex = VectorSearch(
    keyword_fields=["filename"]
)

vindex.fit(
    X,
    chunks
)

query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)
results = vindex.search(
    query_vector,
    num_results=5
)
results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

In [13]:
query = "How do I store vectors in PostgreSQL?"
query_vector = embed.encode(query)
vector_results = vindex.search(
    query_vector,
    num_results=5
)
for r in vector_results:
    print(r["filename"])

02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


In [15]:
from minsearch import Index
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
text_fields=["content"]
index.fit(chunks)

text_results = index.search(
    query,
    num_results=5
)
for r in text_results:
    print(r["filename"])

02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


In [16]:
vector_files = [
    r["filename"]
    for r in vector_results
]


text_files = [
    r["filename"]
    for r in text_results
]

set(vector_files) - set(text_files)

{'02-vector-search/lessons/08-pgvector.md'}

In [17]:
query = "How do I give the model access to tools?"
query_vector = embed.encode(query)
vector_results = vindex.search(
    query_vector,
    num_results=5
)
text_results = index.search(
    query,
    num_results=5
)

In [18]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])

            scores[key] = scores.get(key, 0) + 1 / (k + rank)

            docs[key] = doc

    ranked = sorted(
        scores,
        key=scores.get,
        reverse=True
    )

    return [
        docs[key]
        for key in ranked[:num_results]
    ]

In [19]:
results = rrf(
    [
        vector_results,
        text_results
    ]
)
results[0]["filename"]

'01-agentic-rag/lessons/13-function-calling.md'